# Gahtering & Expolring NOAA Data

## Setting up

To get started we need to pip install the XlsxWriter library as this is not standard to google colab. Then we import the necessary libaries.

In [ ]:
!pip install XlsxWriter

In [ ]:
from google.colab import files
import json
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
import xlsxwriter

## Loading and Processing the data

We create a data frame names "precip" and use the pl.read_csv() function to read our csv data into a data frame, skip_rows=4 skips the rows that do not contains data but contain the data set information, truncate_ragged_lines truncates lines longer than the schema in the data set.

In [ ]:
precip = pl.read_csv('/content/sd_precip.csv', \
                     skip_rows=4, truncate_ragged_lines=True)
print(precip)

We only need to keep the date and value columns so using the select method we grab those two columns and during the process we also cast the date column to a string data type. We then remane the value column to all lowercase.

In [ ]:
precip = precip.select(pl.col('Date').cast(pl.Utf8), pl.col('Value'),)
precip = precip.rename({'Value': 'value'})
print(precip)

We want to split the date column into three separate coloumns one for the month number, one for the year number and we create one for the day number. We then combine these values into a new column called date. We cast the new date column to a date data type and drop the columns created and no longer needed.

In [ ]:
precip = precip.with_columns(pl.col('Date').str.slice(4).alias('month'),)
precip = precip.with_columns(pl.col('Date').str.replace(r"..$", "").alias('year'),)
precip = precip.with_columns(pl.lit("01").alias('day'))
print(precip)

In [ ]:
precip = precip.with_columns((pl.col('year') + "-" + pl.col('month') + \
                              "-" + pl.col('day')).alias('date'),)
print(precip)

In [ ]:
precip = precip.with_columns(pl.col('date').str.to_date())
mo_avg = precip.drop('Date', 'year', 'day')
precip = precip.drop('Date', 'month', 'year', 'day')
precip = precip.select('date', 'value')
print(precip)

## Plotting the data

To visually explore the data we use the lineplot() function from Seaborn and give the date as our x value and the value and the y value.

In [ ]:
sns.lineplot(x='date', y='value', data=precip)

Next lets aggregate the data by gouping by the months and finding the mean, min, and max of each month over the course of the data. We can save this data frame to an excel file and visualize this data to see the monthly average, min, and max over our time period choosen.

In [ ]:
mo_avg = mo_avg.group_by('month').agg(
    pl.col('value').mean().alias('avg_precip'),
    pl.col('value').min().alias('min_precip'),
    pl.col('value').max().alias('max_precip')
)
mo_avg = mo_avg.sort('month')
print(mo_avg)

In [ ]:
mo_avg.write_excel('mo_avg.xlsx')
files.download('mo_avg.xlsx')

In [ ]:
sns.lineplot(data=mo_avg)